In [ ]:

from google.colab import files

uploaded = files.upload()


Saving SwiftRoute_Logistics_Dataset.csv to SwiftRoute_Logistics_Dataset.csv


In [ ]:

import pandas as pd

df = pd.read_csv("SwiftRoute_Logistics_Dataset.csv")

print("Dataset shape:", df.shape)
display(df.head())
print("\nColumn names:")
print(df.columns.tolist())


Dataset shape: (500, 9)


,Shipment_ID,Order_Date,Distance_KM,Shipment_Weight_KG,Shipment_Volume,Delivery_Time_Days,Delivery_Zone,Delivery_Status,Transportation_Cost
0,SH0001,2026-01-01,112,85.02,2,6,North,Delivered,783.58
1,SH0002,2026-01-02,445,14.53,10,3,North,Delayed,2374.18
2,SH0003,2026-01-03,358,71.18,5,1,North,Delayed,2060.69
3,SH0004,2026-01-04,280,55.73,10,1,East,Delayed,1585.57
4,SH0005,2026-01-05,116,30.35,1,1,South,Delayed,799.37



Column names:
['Shipment_ID', 'Order_Date', 'Distance_KM', 'Shipment_Weight_KG', 'Shipment_Volume', 'Delivery_Time_Days', 'Delivery_Zone', 'Delivery_Status', 'Transportation_Cost']


In [ ]:

# Prepar features and target

target = "Delivery_Time_Days"

features = [
    "Distance_KM",
    "Shipment_Weight_KG",
    "Shipment_Volume",
    "Transportation_Cost",
    "Delivery_Zone"
]

X = df[features].copy()
y = df[target].copy()

print("Input features:", X.columns.tolist())
print("Target variable:", target)
print("Feature dataset shape:", X.shape)
print("Target dataset shape:", y.shape)

display(X.head())


Input features: ['Distance_KM', 'Shipment_Weight_KG', 'Shipment_Volume', 'Transportation_Cost', 'Delivery_Zone']
Target variable: Delivery_Time_Days
Feature dataset shape: (500, 5)
Target dataset shape: (500,)


,Distance_KM,Shipment_Weight_KG,Shipment_Volume,Transportation_Cost,Delivery_Zone
0,112,85.02,2,783.58,North
1,445,14.53,10,2374.18,North
2,358,71.18,5,2060.69,North
3,280,55.73,10,1585.57,East
4,116,30.35,1,799.37,South


In [ ]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))


Training records: 400
Testing records: 100


In [ ]:
# Linear Rgression Model

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

numeric_features = [
    "Distance_KM",
    "Shipment_Weight_KG",
    "Shipment_Volume",
    "Transportation_Cost"
]

categorical_features = ["Delivery_Zone"]

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", "passthrough", numeric_features),
        ("zone", OneHotEncoder(handle_unknown="ignore"),
         categorical_features)
    ]
)

linear_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)

print("Linear Regression model trained successfully!")


Linear Regression model trained successfully!


In [ ]:

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Predict delivery times for the test data
y_pred = linear_model.predict(X_test)

# Calculate evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Linear Regression Model Evaluation")
print("----------------------------------")
print(f"Mean Absolute Error (MAE): {mae:.2f} days")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} days")
print(f"R-squared (R²): {r2:.3f}")


Linear Regression Model Evaluation
----------------------------------
Mean Absolute Error (MAE): 1.88 days
Root Mean Squared Error (RMSE): 2.17 days
R-squared (R²): -0.069


In [ ]:
# Decision Tree Model

from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# Create the Decision Tree model
tree_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DecisionTreeRegressor(
        max_depth=4,
        random_state=42
    ))
])

# Train the model
tree_model.fit(X_train, y_train)

# Predict delivery times
tree_pred = tree_model.predict(X_test)

# Evaluate the model
tree_mae = mean_absolute_error(y_test, tree_pred)
tree_rmse = np.sqrt(mean_squared_error(y_test, tree_pred))
tree_r2 = r2_score(y_test, tree_pred)

print("Decision Tree Model Evaluation")
print("--------------------------------")
print(f"MAE: {tree_mae:.2f} days")
print(f"RMSE: {tree_rmse:.2f} days")
print(f"R-squared (R²): {tree_r2:.3f}")


Decision Tree Model Evaluation
--------------------------------
MAE: 1.89 days
RMSE: 2.23 days
R-squared (R²): -0.126


In [ ]:
# Comparing both models

comparison = pd.DataFrame({
    "Model": ["Linear Regression", "Decision Tree"],
    "MAE": [mae, tree_mae],
    "RMSE": [rmse, tree_rmse],
    "R2": [r2, tree_r2]
})

display(comparison.round(3))


,Model,MAE,RMSE,R2
0,Linear Regression,1.876,2.170,-0.069
1,Decision Tree,1.890,2.227,-0.126


In [ ]:
# Hyperparameter Tuning


from sklearn.model_selection import GridSearchCV

# Tune the Decision Tree using training data only
param_grid = {
    "model__max_depth": [2, 3, 4, 5, None],
    "model__min_samples_leaf": [1, 2, 5, 10]
}

grid_search = GridSearchCV(
    estimator=tree_model,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=5,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

best_tree = grid_search.best_estimator_

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation MAE:",
      round(-grid_search.best_score_, 3), "days")


Best parameters: {'model__max_depth': 2, 'model__min_samples_leaf': 10}
Best cross-validation MAE: 1.985 days


In [ ]:
# Evaluating Tune Model


tuned_pred = best_tree.predict(X_test)

tuned_mae = mean_absolute_error(y_test, tuned_pred)
tuned_rmse = np.sqrt(mean_squared_error(y_test, tuned_pred))
tuned_r2 = r2_score(y_test, tuned_pred)

print("Tuned Decision Tree — Test Results")
print(f"MAE: {tuned_mae:.3f} days")
print(f"RMSE: {tuned_rmse:.3f} days")
print(f"R²: {tuned_r2:.3f}")


Tuned Decision Tree — Test Results
MAE: 1.921 days
RMSE: 2.266 days
R²: -0.166
